In [0]:
%run "../../commons/commons_imports"

In [0]:
df_aluno_bronze = read(
    base_path=BRONZE_PATH,
    table_name=TS_ALUNO,
    recursive_by_year=True
)

In [0]:
df_aluno_silver = (
    df_aluno_bronze

    # ============================================
    # Conversão de tipos
    # ============================================

    .withColumn("NU_ANO_AVALIACAO", col("NU_ANO_AVALIACAO").cast("int"))
    .withColumn("CO_UF", col("CO_UF").cast("int"))
    .withColumn("ID_ALUNO", col("ID_ALUNO").cast("long"))
    .withColumn("TP_SERIE", col("TP_SERIE").cast("int"))
    .withColumn("ID_ESCOLA", col("ID_ESCOLA").cast("long"))
    .withColumn("TP_DEPENDENCIA", col("TP_DEPENDENCIA").cast("int"))
    .withColumn("CO_MUNICIPIO", col("CO_MUNICIPIO").cast("int"))

    .withColumn("IN_PRESENCA_LP", col("IN_PRESENCA_LP").cast("int"))
    .withColumn("IN_PREENCHIMENTO_LP", col("IN_PREENCHIMENTO_LP").cast("int"))
    .withColumn("IN_ALFABETIZADO", col("IN_ALFABETIZADO").cast("int"))

    .withColumn("CO_CADERNO_LP", col("CO_CADERNO_LP").cast("int"))

    .withColumn("CO_BLOCO_1", col("CO_BLOCO_1").cast("int"))
    .withColumn("CO_BLOCO_2", col("CO_BLOCO_2").cast("int"))
    .withColumn("CO_BLOCO_3", col("CO_BLOCO_3").cast("int"))
    .withColumn("CO_BLOCO_4", col("CO_BLOCO_4").cast("int"))

    .withColumn("VL_PESO_ALUNO_LP", col("VL_PESO_ALUNO_LP").cast("double"))
    .withColumn("VL_PROFICIENCIA_LP", col("VL_PROFICIENCIA_LP").cast("double"))

    # ============================================
    # Padronização de texto
    # ============================================

    .withColumn(
        "NO_MUNICIPIO",
        initcap(trim(lower(col("NO_MUNICIPIO"))))
    )

    # ============================================
    # Colunas técnicas
    # ============================================

    .withColumn("DT_PROCESSAMENTO", current_date())
    .withColumn("TS_PROCESSAMENTO", current_timestamp())
)

In [0]:


df_aluno_silver = (
    df_aluno_silver
    .withColumn(
        "SK_ALUNO",
        sha2(
            concat_ws(
                "|",
                col("NU_ANO_AVALIACAO"),
                col("ID_ALUNO")
            ),
            256
        )
    )
)

In [0]:
duplicados = (
    df_aluno_silver
    .groupBy(
        "NU_ANO_AVALIACAO",
        "ID_ALUNO"
    )
    .agg(count("*").alias("QTD"))
    .filter(col("QTD") > 1)
)

if duplicados.limit(1).count() > 0:
    raise Exception("Foram encontrados registros duplicados na chave de negócio.")

In [0]:
from pyspark.sql.functions import col

df_aluno_silver_selected = df_aluno_silver.select(

    # ==========================================
    # Chave técnica
    # ==========================================

    "SK_ALUNO",

    # ==========================================
    # Chaves de negócio
    # ==========================================

    "NU_ANO_AVALIACAO",
    "ANO_REFERENCIA",
    "ID_ALUNO",
    "ID_ESCOLA",

    # ==========================================
    # Informações da escola
    # ==========================================

    "CO_UF",
    "SG_UF",
    "CO_MUNICIPIO",
    "NO_MUNICIPIO",
    "TP_DEPENDENCIA",
    "TP_SERIE",

    # ==========================================
    # Informações da prova
    # ==========================================

    "IN_PRESENCA_LP",
    "IN_PREENCHIMENTO_LP",
    "CO_CADERNO_LP",

    "CO_BLOCO_1",
    "TX_RESPOSTA_BLOCO_1",
    "TX_GABARITO_BLOCO_1",

    "CO_BLOCO_2",
    "TX_RESPOSTA_BLOCO_2",
    "TX_GABARITO_BLOCO_2",

    "CO_BLOCO_3",
    "TX_RESPOSTA_BLOCO_3",
    "TX_GABARITO_BLOCO_3",

    "CO_BLOCO_4",
    "TX_RESPOSTA_BLOCO_4",
    "TX_GABARITO_BLOCO_4",

    # ==========================================
    # Resultados
    # ==========================================

    "VL_PESO_ALUNO_LP",
    "VL_PROFICIENCIA_LP",
    "IN_ALFABETIZADO",

    # ==========================================
    # Auditoria
    # ==========================================

    "DT_PROCESSAMENTO",
    "TS_PROCESSAMENTO"
)

In [0]:
write_delta(
    df=df_aluno_silver_selected,
    base_path=SILVER_PATH,
    table_name=TS_ALUNO,
    merge_keys=["SK_ALUNO"]
)